# Road Accident Ai Alerting System

## Dataset:
https://huggingface.co/datasets/M-ArslanArshad/Car_Accidents_and_deformation_dataset

### Data Overview

> The Car Accidents and Deformation Dataset is a high-quality, manually curated collection of real-world car accident images, fully collected and annotated by the author. The dataset is intended to support machine learning tasks focused on:

> 1. Vehicle damage classification
> 2. Deformation severity estimation
> 3. Intelligent transportation systems
> 4. Insurance automation and incident reporting

### Dataset Structure
> The dataset contains two main folders: images/ and labels/, each with train/ and val/ subfolders:

```text
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
```





In [1]:
from pathlib import Path
import os
from huggingface_hub import snapshot_download

k:\Ali Turab Coding Space\Car_Accidents_and_deformation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_path = f"{Path().cwd()}/dataset"
print(dataset_path)
os.makedirs(dataset_path, exist_ok=True)

dataset_path = snapshot_download(
    repo_id="M-ArslanArshad/Car_Accidents_and_deformation_dataset",
    repo_type="dataset",
    local_dir=dataset_path,
)

print("Downloaded to:", dataset_path)

k:\Ali Turab Coding Space\Car_Accidents_and_deformation/dataset


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 213.67it/s]

Downloaded to: K:\Ali Turab Coding Space\Car_Accidents_and_deformation\dataset


#### Note: Following Code is to Unzip Data nothting else 

In [3]:
import os
from pathlib import Path
import shutil
import subprocess
import rarfile

# Point rarfile to WinRAR / 7-Zip on Windows (or unrar on Linux)
possible_tools = [
    r"C:\Program Files\WinRAR\UnRAR.exe",
    r"C:\Program Files (x86)\WinRAR\UnRAR.exe",
    r"C:\Program Files\7-Zip\7z.exe",
    shutil.which("unrar"),
    shutil.which("7z"),
]

tool_found = next((t for t in possible_tools if t and os.path.exists(t)), None)

# Dataset archive & destination
rar_path = Path("dataset/car_accident_severity.rar")
output_dir = Path("dataset/car_accident_severity")

if not rar_path.exists():
  raise FileNotFoundError(f"Dataset not found: {rar_path}")

output_dir.mkdir(parents=True, exist_ok=True)

if tool_found and "7z" in tool_found.lower():
  # Use 7-Zip directly if WinRAR is not installed
  subprocess.run(
      [tool_found, "x", str(rar_path), f"-o{output_dir}", "-y"], check=True
  )
else:
  if tool_found:
    rarfile.UNRAR_TOOL = tool_found
  with rarfile.RarFile(rar_path) as archive:
    files = archive.infolist()
    print(f"Total archive entries: {len(files)}")
    archive.extractall(path=output_dir)

print("Dataset extraction completed!")
print(f"Extracted to: {output_dir.resolve()}")


Total archive entries: 2453
Dataset extraction completed!
Extracted to: K:\Ali Turab Coding Space\Car_Accidents_and_deformation\dataset\car_accident_severity


In [4]:
#checking for Dataset and Data
import pandas as pd

print(f"{dataset_path}/car_accident_severity/custom_dataset/images/train")
print(f"{dataset_path}/car_accident_severity/custom_dataset/labels/train")

K:\Ali Turab Coding Space\Car_Accidents_and_deformation\dataset/car_accident_severity/custom_dataset/images/train
K:\Ali Turab Coding Space\Car_Accidents_and_deformation\dataset/car_accident_severity/custom_dataset/labels/train


Each .txt label file follows YOLO format:
<class_id> <x_center> <y_center> <width> <height> (all normalized)

Example:
```text
4 0.560967 0.598094 0.829009 0.629283
```

## Class Labels

| Class ID | Description | Estimated Deformity |
| :---: | :--- | :---: |
| 0 | No Accident | 0% |
| 1 | Minor Accident | ~30% |
| 2 | Moderate Accident | ~50% |
| 3 | Severe Accident | ~70% |
| 4 | Totaled Vehicle | ~100% (on fire, flipped, crushed) |


## Cleaning and making Data Perfect to use in Model

Counting Data and Creating Dataframe that will have images and there boxes in the Dataset to train model Perfectly.

In [12]:
splits = [
    "dataset/car_accident_severity/custom_dataset/images/train",
    "dataset/car_accident_severity/custom_dataset/images/val",
]

for path_str in splits:
  folder = Path(path_str)
  count = len([f for f in folder.iterdir() if f.is_file()])
  print(f"{path_str}: {count} files")

dataset/car_accident_severity/custom_dataset/images/train: 977 files
dataset/car_accident_severity/custom_dataset/images/val: 246 files


In [ ]:
train_images_folder = Path("dataset/car_accident_severity/custom_dataset/images/train")

images_data = [f.name for f in train_images_folder.iterdir() if f.is_file()]
print(len(images_data))

# train_df = pd.DataFrame(images_data, columns=["fileName"])

# display(train_df)

977


,fileName
0,10_flip70.jpg
1,11_flip70.jpg
2,123.jpg
3,123_flip50.jpg
4,12_flip50.jpg
...,...
972,xzza.jpg
973,ybyb.jpg
974,yhguuytfgh.jpg
975,yjghsyjdsh.jpg


In [ ]:
dataSetPath = "./dataset/car_accident_severity/custom_dataset/labels/"

data = []

def get_image_data(imageName, type = "train"):
    fileName = imageName.split(".")[0]
    if f"{dataSetPath}/{type}/{fileName}.txt":
        lines = Path(f"{dataSetPath}/{type}/{fileName}.txt").read_text().splitlines()
        for b in lines:
            data.append({
                "fileName" : fileName,
                "imageName": imageName,
                "data": b
            })

In [81]:
for image in images_data:
    get_image_data(image)

train_df = pd.DataFrame(data)
display(train_df)

,fileName,imageName,data
0,10_flip70,10_flip70.jpg,4 0.560967 0.598094 0.829009 0.629283
1,11_flip70,11_flip70.jpg,4 0.568632 0.316635 0.240566 0.400844
2,11_flip70,11_flip70.jpg,4 0.252005 0.543709 0.491745 0.867784
3,123,123.jpg,2 0.518826 0.465313 0.962348 0.917090
4,123_flip50,123_flip50.jpg,2 0.472847 0.467857 0.921580 0.760714
...,...,...,...
5953,yjghsyjdsh,yjghsyjdsh.jpg,3 0.737125 0.588235 0.156250 0.205147
5954,yjghsyjdsh,yjghsyjdsh.jpg,0 0.877125 0.583824 0.136250 0.143382
5955,yjghsyjdsh,yjghsyjdsh.jpg,0 0.617125 0.517647 0.053750 0.050735
5956,yutrew,yutrew.jpg,2 0.599000 0.619118 0.242500 0.156618


In [82]:
grouped_df = (
    train_df.groupby("imageName")
    .agg({
        "data": list
    })
    .reset_index()
)

print(grouped_df.head())

        imageName                                               data
0   10_flip70.jpg  [4 0.560967 0.598094 0.829009 0.629283, 4 0.56...
1   11_flip70.jpg  [4 0.568632 0.316635 0.240566 0.400844, 4 0.25...
2         123.jpg  [2 0.518826 0.465313 0.962348 0.917090, 2 0.51...
3  123_flip50.jpg  [2 0.472847 0.467857 0.921580 0.760714, 2 0.47...
4   12_flip50.jpg  [2 0.496924 0.533036 0.991133 0.933928, 2 0.49...


In [84]:
def parse_annotations(annotations):
    boxes = []
    labels = []

    for annotation in annotations:
        values = annotation.split()

        class_id = int(values[0])
        x_center, y_center, width, height = map(float, values[1:])

        labels.append(class_id)

        boxes.append([
            x_center,
            y_center,
            width,
            height
        ])

    return labels, boxes


grouped_df[["labels", "boxes"]] = grouped_df["data"].apply(
    lambda x: pd.Series(parse_annotations(x))
)

grouped_df

,imageName,data,labels,boxes
0,10_flip70.jpg,"[4 0.560967 0.598094 0.829009 0.629283, 4 0.56...","[4, 4]","[[0.560967, 0.598094, 0.829009, 0.629283], [0...."
1,11_flip70.jpg,"[4 0.568632 0.316635 0.240566 0.400844, 4 0.25...","[4, 4, 4, 4]","[[0.568632, 0.316635, 0.240566, 0.400844], [0...."
2,123.jpg,"[2 0.518826 0.465313 0.962348 0.917090, 2 0.51...","[2, 2]","[[0.518826, 0.465313, 0.962348, 0.91709], [0.5..."
3,123_flip50.jpg,"[2 0.472847 0.467857 0.921580 0.760714, 2 0.47...","[2, 2]","[[0.472847, 0.467857, 0.92158, 0.760714], [0.4..."
4,12_flip50.jpg,"[2 0.496924 0.533036 0.991133 0.933928, 2 0.49...","[2, 2]","[[0.496924, 0.533036, 0.991133, 0.933928], [0...."
...,...,...,...,...
972,xzza.jpg,"[1 0.567845 0.430153 0.864310 0.652470, 1 0.56...","[1, 1]","[[0.567845, 0.430153, 0.86431, 0.65247], [0.56..."
973,ybyb.jpg,"[4 0.808803 0.571066 0.382394 0.367174, 4 0.43...","[4, 4, 4, 4]","[[0.808803, 0.571066, 0.382394, 0.367174], [0...."
974,yhguuytfgh.jpg,"[3 0.730750 0.587134 0.538500 0.438259, 3 0.25...","[3, 3, 3, 3]","[[0.73075, 0.587134, 0.5385, 0.438259], [0.252..."
975,yjghsyjdsh.jpg,"[3 0.737125 0.588235 0.156250 0.205147, 0 0.87...","[3, 0, 0, 3, 0, 0]","[[0.737125, 0.588235, 0.15625, 0.205147], [0.8..."


In [ ]:
import torch

print("PyTorch Version:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())

print("CUDA Version:", torch.version.cuda)

print("GPU Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

    print("Current Device:", torch.cuda.current_device())

    print("GPU Memory:", round( torch.cuda.get_device_properties(0).total_memory / 1024**3, 2 ), "GB")

PyTorch Version: 2.15.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Count: 1
GPU Name: NVIDIA GeForce RTX 3060
Current Device: 0
GPU Memory: 12.0 GB
